# Configuración

In [1]:
import sys
import os

# Agregar carpeta raíz del proyecto
sys.path.append(
    os.path.abspath("..")
)

# Cargar datos

In [2]:
from src.cargar_datos import cargar_datos

personal, calendario, configuracion_puestos = cargar_datos(
    "../data/cerebro_farallones.xlsx"
)

print("Personal:", len(personal))
print("Fechas calendario:", len(calendario))

Personal: 88
Fechas calendario: 31


# Crear turnos

In [3]:
from src.crear_turnos import crear_turnos

turnos_df = crear_turnos(
    calendario,
    configuracion_puestos
)

display(turnos_df)

,id,puesto,tipo,fecha,fecha_inicio,fecha_fin,duracion_dias,horario,personas,tipo_dia,festivo
0,AP-01,Amor y Paz,bloque,2026-10-01,2026-10-01,2026-10-05,5,24h,4,NaN,NaN
1,PL-01,Pato-Leonera,bloque,2026-10-01,2026-10-01,2026-10-05,5,24h,4,NaN,NaN
2,PP-01,Pato-Pance,turno,2026-10-03,2026-10-03,2026-10-03,1,dia,5,sábado,SI
3,TO-01,Topacio,turno,2026-10-03,2026-10-03,2026-10-03,1,dia,1,sábado,SI
4,PP-02,Pato-Pance,turno,2026-10-04,2026-10-04,2026-10-04,1,dia,5,domingo,SI
5,TO-02,Topacio,turno,2026-10-04,2026-10-04,2026-10-04,1,dia,1,domingo,SI
6,AP-02,Amor y Paz,bloque,2026-10-05,2026-10-05,2026-10-09,5,24h,4,NaN,NaN
7,PL-02,Pato-Leonera,bloque,2026-10-05,2026-10-05,2026-10-09,5,24h,4,NaN,NaN
8,AP-03,Amor y Paz,bloque,2026-10-09,2026-10-09,2026-10-13,5,24h,4,NaN,NaN
9,PL-03,Pato-Leonera,bloque,2026-10-09,2026-10-09,2026-10-13,5,24h,4,NaN,NaN


# Resumen de puestos

In [4]:
print("\n=== RESUMEN DE TURNOS ===")

print(
    turnos_df.groupby(
        "puesto"
    ).agg(
        turnos=("id", "count"),
        personas=("personas", "sum")
    )
)


=== RESUMEN DE TURNOS ===
              turnos  personas
puesto                        
Amor y Paz         7        28
Pato-Leonera       7        28
Pato-Pance        10        50
Topacio           10        10


# Crear modelo

In [5]:
from src.modelo import construir_modelo_base

model, x = construir_modelo_base(
    personal,
    turnos_df
)

print("Modelo base creado correctamente.")

Modelo base creado correctamente.


# Restricciones específicas

In [6]:
from src.restricciones_bloques import (
    agregar_restricciones_bloques
)

from src.restricciones_puestos import (
    agregar_restricciones_puestos
)


agregar_restricciones_bloques(
    model,
    x,
    personal,
    turnos_df
)


agregar_restricciones_puestos(
    model,
    x,
    personal,
    turnos_df
)

print("Restricciones agregadas correctamente.")

Restricciones agregadas correctamente.


# Función objetivo y balance de carga

In [7]:
from src.objetivo import (
    agregar_funcion_objetivo
)


carga = agregar_funcion_objetivo(

    model=model,

    x=x,

    personal=personal,

    turnos_df=turnos_df,

    semilla=None

)

print("Función objetivo agregada correctamente.")

Función objetivo agregada correctamente.


# Resolver modelo

In [8]:
from ortools.sat.python import cp_model


solver = cp_model.CpSolver()

status = solver.Solve(model)

print(
    "Estado:",
    solver.StatusName(status)
)

Estado: OPTIMAL


# Validar solución

In [9]:
from src.validar_solucion import (
    validar_solucion
)


if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
]:

    errores = validar_solucion(
        solver,
        x,
        personal,
        turnos_df
    )

else:

    errores = [
        "No se encontró solución."
    ]


VALIDACIÓN DE LA SOLUCIÓN

✓ SOLUCIÓN VÁLIDA
Todas las reglas verificadas se cumplen.


# Reportes

In [10]:
from src.reportes import (
    imprimir_asignaciones,
    crear_resumen_carga
)


if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
]:

    imprimir_asignaciones(
        solver,
        x,
        personal,
        turnos_df
    )

    resumen_df = crear_resumen_carga(
        solver,
        x,
        personal,
        turnos_df
    )

    display(resumen_df)

else:

    print(
        "No se encontró solución."
    )


=== ASIGNACIONES ===

AP-01 | 2026-10-01 - 2026-10-05 | Amor y Paz | 24h
- Diana Ramos
- Maria Fernanda Parra
- Samuel Barona
- Sebastian Ovalle

PL-01 | 2026-10-01 - 2026-10-05 | Pato-Leonera | 24h
- Cristian Libreros
- Esmeralda Acosta
- Miguel Castro
- Monica Patricia Ramirez Lopez

PP-01 | 2026-10-03 - 2026-10-03 | Pato-Pance | dia
- Alexander Gomez
- Dayro Riaños
- Edgar Reyes
- Luz Dalia Miranda
- Samuel Morales 

TO-01 | 2026-10-03 - 2026-10-03 | Topacio | dia
- Alejandra Garcia

PP-02 | 2026-10-04 - 2026-10-04 | Pato-Pance | dia
- Alexander Morales
- Andres De Los Rios
- Danny Leandro Mora
- David Castaño
- Marino Lasso

TO-02 | 2026-10-04 - 2026-10-04 | Topacio | dia
- Lina Pareja

AP-02 | 2026-10-05 - 2026-10-09 | Amor y Paz | 24h
- Carlos Perea
- Juliana Cerón
- Wilfrido Ibarbo
- Karen Alvarado

PL-02 | 2026-10-05 - 2026-10-09 | Pato-Leonera | 24h
- Cesar Rueda
- Paola Alzate
- Sandra Milena Villada
- Stiven Atoy

AP-03 | 2026-10-09 - 2026-10-13 | Amor y Paz | 24h
- Danny L

,nombre,cargo,estrategia,activo,conductor,amor_y_paz,pato_leonera,pato_pance,topacio,total_puestos
0,Alexander Gomez,TECNICO,ECOTURISMO,SI,CARRO y MOTO,0,5,2,0,7
1,Juan Manuel Guzman,TECNICO,FUNCIONARIO,SI,CARRO,0,5,2,0,7
2,Alice Cadena,TECNICO,FUNCIONARIO,SI,NO,0,5,1,1,7
3,Zoraida Bermudez Cardona,OPERARIO,RESTAURACION,SI,NO,5,0,2,0,7
4,Diana Ramos,OPERARIO,RESTAURACION,SI,NO,5,0,1,1,7
...,...,...,...,...,...,...,...,...,...,...
83,Edileunis Beatriz Pitre Solano,PROFESIONAL,ADMINISTRATIVO,NO,NO,0,0,0,0,0
84,Viviana urbano,TECNICO,PVC,NO,NO,0,0,0,0,0
85,Lina Yajaira Pelaez Celada,PROFESIONAL,PVC,NO,NO,0,0,0,0,0
86,Dayana Marcela Alegria,PROFESIONAL,PVC,NO,NO,0,0,0,0,0


# Resumen por cargo

In [11]:
if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
]:

    resumen_cargo = (
        resumen_df
        .groupby("cargo")
        .agg(
            personas=("nombre", "count"),
            puestos_totales=("total_puestos", "sum"),
            promedio_puestos=(
                "total_puestos",
                "mean"
            ),
            minimo=("total_puestos", "min"),
            maximo=("total_puestos", "max")
        )
        .round(2)
    )

    display(resumen_cargo)

,personas,puestos_totales,promedio_puestos,minimo,maximo
cargo,,,,,
OPERARIO,20,92,4.60,0,7
PROFESIONAL,38,123,3.24,0,5
TECNICO,22,99,4.50,0,7
TECNOLOGO,8,26,3.25,0,6


# Exportar Excel

In [12]:
from src.exportar_resultados import (
    exportar_excel
)


if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
] and len(errores) == 0:

    exportar_excel(
        solver=solver,
        x=x,
        personal=personal,
        turnos_df=turnos_df,
        archivo_salida="../outputs/asignacion_octubre.xlsx"
    )

    print(
        "\n✓ Archivo exportado correctamente."
    )


✓ Excel exportado:
../outputs/asignacion_octubre.xlsx

✓ Archivo exportado correctamente.
